<a href="https://colab.research.google.com/github/rayaneghilene/PIDA-M2-Courses/blob/main/Lecture_5_Guided_Web_Scraping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lecture 5 — Introduction to Web Scraping Basics

**Master 2 PIDA**  



This notebook demonstrates step by step how to:
1. **Scrape** textual data from a simple site ([quotes.toscrape.com](https://quotes.toscrape.com)) using `requests` and `BeautifulSoup`.
2. Build a structured dataset with `pandas`.
3. Perform **sentiment analysis** on the scraped quotes with `TextBlob`.
4. Explore how sentiment and topic tags change **across pages** (as a proxy for time).
5. Visualise the results with **histograms, bar plots, and area charts**.

**How to work:** run cells from top to bottom with **Shift+Enter.** Each “Pause” asks you to predict or write an answer *before* running the next cell. You may edit and rerun a cell. You do not need to memorize Python syntax, just understand what the code is doing.

> This notebook is fully **annotated** for beginners: read the Markdown cells carefully before each code cell.



## 1. Installing required libraries

You need these Python libraries:
- `requests` to download web pages
- `beautifulsoup4` to parse HTML
- `pandas` for handling tabular data
- `textblob` for simple sentiment analysis
- `matplotlib` and `seaborn` for plotting

In most notebook environments (e.g. Colab), you can run the following cell **once** to install them.


In [ ]:
!pip install requests beautifulsoup4 pandas textblob matplotlib seaborn


## 2. Importing libraries

After installing, we import them to use in Python.


In [ ]:

import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
from textblob import TextBlob
import matplotlib.pyplot as plt
import seaborn as sns



## 3. Preparing to scrape

We define:
- `base_url` as a template for each page.
- an empty list `quotes` to collect all results as dictionaries.


In [ ]:

base_url = "https://quotes.toscrape.com/page/{}/"
quotes = []



## 4. Scraping the website

We loop over pages 1–10:
1. Download each page with `requests.get()`.
2. Stop the loop if we get an error code or if there are no quotes.
3. Parse the HTML with `BeautifulSoup`.
4. Find all the blocks containing quotes.
5. For each quote:
   - Extract the text, author, and tags.
   - Compute a simple sentiment polarity using `TextBlob`.
   - Save everything into our list as a dictionary.


In [ ]:

for page in range(1, 11):  # The site has 10 pages
    url = base_url.format(page)

    # requests.get() retrieves the page.
    # r stores the response.

    r = requests.get(url)

    if r.status_code != 200:
        break
    ## Status code is like page not found error 404. Status code == 200 means
    ## workflow done successefully

    soup = BeautifulSoup(r.text, "html.parser")

    ## Soup will structure all your extracted data.

    quote_blocks = soup.find_all("div", class_="quote")
    if not quote_blocks:
        break

    for qb in quote_blocks:
        text = qb.find("span", class_="text").get_text(strip=True)
        author = qb.find("small", class_="author").get_text(strip=True)
        tags = [tag.get_text() for tag in qb.find_all("a", class_="tag")]
        polarity = TextBlob(text).sentiment.polarity

        quotes.append({
            "page": page,
            "text": text,
            "author": author,
            "tags": tags,
            "sentiment": polarity
        })


## Store the results in a dataframe.
df = pd.DataFrame(quotes)

## Inspect your dataframe

Remember what we discussed last week
```
df.shape

df.info()

df.isna().sum()
```

In [ ]:
## TODO: Add your data inspection code here

...


## 5. Sentiment distribution

We visualise the distribution of sentiment polarity (between -1 and +1) across all quotes.


In [ ]:

plt.figure(figsize=(8,5))
sns.histplot(df["sentiment"], bins=20, kde=True)
plt.title("Distribution of Sentiment Scores for Quotes")
plt.xlabel("Sentiment Polarity (-1=Negative, +1=Positive)")
plt.ylabel("Number of Quotes")
plt.show()



## 6. Average sentiment by author

We group by `author`, compute the **mean sentiment** per author, and visualise it as a barplot.


In [ ]:
## TODO: Replace the three dots ... with your code
### Remember here you should group by author and then compute the mean sentiment

author_sentiment = df.groupby( ... )[ ... ].mean().sort_values()

plt.figure(figsize=(8,6))
sns.barplot(y=author_sentiment.index, x=author_sentiment.values, palette="coolwarm")
plt.axvline(0, color="black", linestyle="--")
plt.title("Average Sentiment per Author")
plt.xlabel("Average Sentiment Polarity")
plt.ylabel("Author")
plt.show()



## 7. Filtering to authors with ≥3 quotes

This makes the chart clearer and comparisons more meaningful.


In [ ]:

author_counts = df['author'].value_counts()

## TODO: Replace the three dots ... with your code
## Here you want to select the authors with at least 3 quotes

authors_to_keep = author_counts[author_counts >= ... ].index

author_sentiment = (
    df[df['author'].isin(authors_to_keep)]
      .groupby('author')[ ... ]
      .mean()
      .sort_values()
)

plt.figure(figsize=(8,6))
sns.barplot(y=author_sentiment.index, x=author_sentiment.values, palette="coolwarm")
plt.axvline(0, color="black", linestyle="--")
plt.title("Average Sentiment (authors with ≥3 quotes)")
plt.xlabel("Average Sentiment Polarity")
plt.ylabel("Author")
plt.tight_layout()
plt.show()



## 8. Sentiment over pages (as proxy for time)

We create a **binary sentiment label** (`positive` or `negative`) and track its proportion per page.
We then visualise it as a **stacked area chart**.


In [ ]:
df['sentiment_label'] = np.where(df['sentiment'] >= 0, 'positive', 'negative')

sentiment_by_page = (
    df.groupby(['page', 'sentiment_label'])
      .size()
      .reset_index(name='count')
)

sentiment_by_page['proportion'] = (
    sentiment_by_page.groupby('page')['count']
                     .transform(lambda x: x / x.sum())
)

pivot_sentiment = sentiment_by_page.pivot(index='page',
                                          columns='sentiment_label',
                                          values='proportion').fillna(0)

for col in ['positive','negative']:
    if col not in pivot_sentiment.columns:
        pivot_sentiment[col] = 0

plt.figure(figsize=(10,6))
plt.stackplot(pivot_sentiment.index,
              pivot_sentiment['positive'],
              pivot_sentiment['negative'],
              labels=['Positive','Negative'],
              colors=['#4CAF50','#F44336'],
              alpha=0.7)
plt.title("Positive vs Negative Quotes by Page")
plt.xlabel("Page")
plt.ylabel("Proportion of quotes")
plt.ylim(0,1)
plt.legend(loc='upper right')
plt.grid(True)
plt.show()



## 9. Top tags over pages

We focus on the **top 5 most frequent tags** and track their relative proportions by page in a **stacked area chart**.


In [ ]:

df_tags = df.explode('tags')

top_tags = df_tags['tags'].value_counts().head(5).index
df_tags_top = df_tags[df_tags['tags'].isin(top_tags)]

tag_counts = (
    df_tags_top.groupby(['page','tags'])
               .size()
               .reset_index(name='count')
)

tag_counts['proportion'] = (
    tag_counts.groupby('page')['count']
              .transform(lambda x: x / x.sum())
)

pivot_tags = tag_counts.pivot(index='page',
                              columns='tags',
                              values='proportion').fillna(0)

plt.figure(figsize=(12,6))
plt.stackplot(pivot_tags.index,
              *[pivot_tags[col] for col in pivot_tags.columns],
              labels=pivot_tags.columns,
              alpha=0.8)

plt.title("Proportion of Top 5 Tags across Pages")
plt.xlabel("Page")
plt.ylabel("Proportion of tags in that page")
plt.ylim(0,1)
plt.legend(loc='upper right')
plt.grid(True)
plt.show()


# Before leaving: Today's assignment : **Write a research finding**

A finding should include:
- What you collected.
- A numerical result.
- What the result means in your sample.
- A limitation.




## Remember
- Sentiment analysis here is **simplistic**; it’s good for illustration but not for serious linguistic research.
- The stacked area charts show how **composition changes** over pages for sentiment and tags.
- In real-world projects, always:
  - check the site’s *robots.txt* and Terms of Service,
  - consider using an API if available,
  - think critically about the meaning of sentiment scores.